## Student-style Jupyter notebook for Albania SDG 8 analysis

In \[1\]:

<span class="comment">\# hw2 - SDG 8 analysis \# Andi Lata \# ok so the
plan is: 5 research questions, stats + probability for each \# let me
just get all imports out of the way first</span> <span
class="keyword">import</span> pandas <span class="keyword">as</span> pd
<span class="keyword">import</span> numpy <span
class="keyword">as</span> np <span class="keyword">import</span>
matplotlib.pyplot <span class="keyword">as</span> plt <span
class="keyword">from</span> scipy <span class="keyword">import</span>
stats <span class="keyword">from</span> scipy.stats <span
class="keyword">import</span> norm, ttest_1samp, ttest_ind, pearsonr
<span class="comment">\# honestly not sure if i need all of these but
better safe</span>

In \[2\]:

<span class="comment">\# load the cleaned data files \# cleaned these in
step 1 - wages, youth unemployment, tourism receipts, etc.</span>
df_wages = pd.read_csv(<span
class="string">"cleaned/wages_clean.csv"</span>) df_youth =
pd.read_csv(<span
class="string">"cleaned/youth_unemployment_clean.csv"</span>) df_tourism
= pd.read_csv(<span
class="string">"cleaned/tourism_receipts_clean.csv"</span>) df_informal
= pd.read_csv(<span
class="string">"cleaned/informal_employment_clean.csv"</span>) df_emp =
pd.read_csv(<span
class="string">"cleaned/tourism_employment_clean.csv"</span>) df_gdp =
pd.read_csv(<span
class="string">"cleaned/gdp_per_capita_clean.csv"</span>) <span
class="comment">\# CPI / inflation -- this one was annoying because
world bank format \# has years as columns so need to melt it</span>
df_cpi_raw = pd.read_csv(<span class="string">"inflation.csv"</span>,
skiprows=4) df_cpi_raw.columns = df_cpi_raw.columns.str.replace(<span
class="string">" "</span>, <span class="string">"\_"</span>) alb_cpi =
df_cpi_raw\[df_cpi_raw\[<span class="string">"Country_Code"</span>\] ==
<span class="string">"ALB"</span>\].copy() year_cols = \[str(y) <span
class="keyword">for</span> y <span class="keyword">in</span> range(2010,
2025)\] alb_cpi = alb_cpi\[\[<span class="string">"Country_Name"</span>,
<span class="string">"Country_Code"</span>\] + year_cols\] alb_cpi =
alb_cpi.melt(id_vars=\[<span class="string">"Country_Name"</span>, <span
class="string">"Country_Code"</span>\], var_name=<span
class="string">"Year"</span>, value_name=<span
class="string">"Inflation_Rate_Pct"</span>) alb_cpi\[<span
class="string">"Year"</span>\] = alb_cpi\[<span
class="string">"Year"</span>\].astype(int) alb_cpi =
alb_cpi.dropna().sort_values(<span
class="string">"Year"</span>).reset_index(drop=True) print(<span
class="string">"all data loaded."</span>)

all data loaded.

In \[3\]:

<span class="comment">\##
────────────────────────────────────────────────────────── \## RQ1:
Formality vs Growth \## Q: as tourism revenue doubled post-2020, did
formal employment \## grow or did informality just absorb the boom? \##
────────────────────────────────────────────────────────── \# H0:
informality rate is the same pre vs post boom \# H1: informality is NOT
significantly lower despite the boom \# (i.e. the boom didnt convert
into real formal jobs)</span> alb_tourism =
df_tourism\[df_tourism\[<span class="string">"Country_Code"</span>\] ==
<span class="string">"ALB"</span>\].copy() alb_informal =
df_informal\[df_informal\[<span class="string">"Country_Code"</span>\]
== <span class="string">"ALB"</span>\].copy() receipts =
alb_tourism\[<span
class="string">"Tourism_Receipts_USD_Billions"</span>\].values <span
class="comment">\# --- distribution plot ---</span>
plt.figure(figsize=(8, 4)) plt.hist(receipts, bins=8, edgecolor=<span
class="string">"black"</span>, color=<span
class="string">"steelblue"</span>, alpha=0.8)
plt.axvline(receipts.mean(), color=<span class="string">"red"</span>,
linestyle=<span class="string">"--"</span>, label=<span
class="string">f"mean = \${receipts.mean():.2f}B"</span>)
plt.title(<span class="string">"RQ1: Distribution of Albania Tourism
Receipts (2010–2024)"</span>) plt.xlabel(<span class="string">"Tourism
Receipts (USD Billions)"</span>) plt.ylabel(<span
class="string">"Frequency"</span>) plt.legend() plt.tight_layout()
plt.show() <span class="comment">\# right-skewed because most years were
low -- the boom years pull the mean up</span> print(<span
class="string">"distribution: right-skewed -- most years were low, boom
years pull the mean up"</span>)

In \[4\]:

<span class="comment">\# probability modeling -- what was the historical
probability of a 3B+ year? \# turns out its pretty low, which makes the
3 consecutive hits even more significant</span> mu = receipts.mean()
sigma = receipts.std() prob_boom = 1 - norm.cdf(3.0, mu, sigma)
print(<span class="string">f"\nprobability modeling:"</span>)
print(<span class="string">f" mean receipts: \${mu:.2f}B \| std:
\${sigma:.2f}B"</span>) print(<span class="string">f" P(receipts \>
\$3B) = {prob_boom:.4f} ({prob_boom\*100:.1f}%)"</span>) print(<span
class="string">f" historically rare -- but Albania hit it 3 years in a
row post-2022"</span>) <span class="comment">\# pdf + cdf side by
side</span> x = np.linspace(0, 7, 200) fig, (ax1, ax2) = plt.subplots(1,
2, figsize=(10, 4)) ax1.plot(x, norm.pdf(x, mu, sigma), color=<span
class="string">"steelblue"</span>, linewidth=2) ax1.axvline(3.0,
color=<span class="string">"red"</span>, linestyle=<span
class="string">"--"</span>, label=<span class="string">"\$3B boom
threshold"</span>) ax1.fill_between(x, norm.pdf(x, mu, sigma), where=(x
\> 3.0), alpha=0.3, color=<span class="string">"red"</span>, label=<span
class="string">f"P={prob_boom:.2f}"</span>) ax1.set_title(<span
class="string">"RQ1: Tourism Receipts PDF"</span>) ax1.set_xlabel(<span
class="string">"USD Billions"</span>) ax1.legend() ax2.plot(x,
norm.cdf(x, mu, sigma), color=<span class="string">"steelblue"</span>,
linewidth=2) ax2.axvline(3.0, color=<span class="string">"red"</span>,
linestyle=<span class="string">"--"</span>, label=<span
class="string">"\$3B threshold"</span>) ax2.set_title(<span
class="string">"RQ1: Tourism Receipts CDF"</span>) ax2.set_xlabel(<span
class="string">"USD Billions"</span>) ax2.legend() plt.tight_layout()
plt.show()

In \[5\]:

<span class="comment">\# hypothesis test: did formality actually improve
during the boom? \# split into pre-boom (before 2020) and post-boom
(2021+)</span> pre_boom_inf = alb_informal\[alb_informal\[<span
class="string">"Year"</span>\] \<= 2019\]\[<span
class="string">"Informal_Employment_Rate_Pct"</span>\].values
post_boom_inf = alb_informal\[alb_informal\[<span
class="string">"Year"</span>\] \>= 2021\]\[<span
class="string">"Informal_Employment_Rate_Pct"</span>\].values
print(<span class="string">f"pre-boom: {pre_boom_inf} --\> mean =
{pre_boom_inf.mean():.2f}%"</span>) print(<span
class="string">f"post-boom: {post_boom_inf} --\> mean =
{post_boom_inf.mean():.2f}%"</span>) t_stat, p_val =
ttest_ind(pre_boom_inf, post_boom_inf) print(<span
class="string">f"\nt-stat: {t_stat:.4f} \| p-value: {p_val:.4f}"</span>)
<span class="comment">\# p \> 0.05 so we fail to reject -- informality
didn't meaningfully budge</span> <span class="keyword">if</span> p_val
\< 0.05: print(<span class="string">"result: REJECT H0 -- informality
changed significantly"</span>) <span class="keyword">else</span>:
print(<span class="string">"result: FAIL TO REJECT H0 -- informality did
not significantly change"</span>) print(<span
class="string">"interpretation: boom ran through the informal economy,
not formal jobs"</span>)

pre-boom: \[33.5 31.9\] --\> mean = 32.70% post-boom: \[31.2 30.8 30.3\]
--\> mean = 30.77% t-stat: 2.8245 \| p-value: 0.0665 result: FAIL TO
REJECT H0 -- informality did not significantly change interpretation:
boom ran through the informal economy, not formal jobs

In \[6\]:

<span class="comment">\##
────────────────────────────────────────────────────────── \## RQ2: Wage
Gap -- does hospitality sector keep up with national avg? \## \## using
CEIC/INSTAT data for accommodation + food wages \## 2021: 29,858 ALL \|
2022: 32,784 ALL \## national avg from the wages_clean dataset \##
──────────────────────────────────────────────────────────</span>
national_2022 = df_wages\[df_wages\[<span class="string">"Year"</span>\]
== 2022\]\[<span
class="string">"Avg_Monthly_Gross_Wage_ALL"</span>\].mean() gap_2021 =
(29858 / 60666) \* 100 <span class="comment">\# 2021 national avg from
INSTAT</span> gap_2022 = (32784 / national_2022) \* 100 print(<span
class="string">f"hospitality as % of national avg:"</span>) print(<span
class="string">f" 2021: {gap_2021:.1f}% \| 2022:
{gap_2022:.1f}%"</span>) print(<span class="string">f" consistently
around 50% -- gap did not close during the boom"</span>) <span
class="comment">\# distribution of all national wages</span> all_wages =
df_wages\[<span
class="string">"Avg_Monthly_Gross_Wage_ALL"</span>\].values
plt.figure(figsize=(8, 4)) plt.hist(all_wages, bins=8, edgecolor=<span
class="string">"black"</span>, color=<span
class="string">"steelblue"</span>, alpha=0.8)
plt.axvline(all_wages.mean(), color=<span class="string">"red"</span>,
linestyle=<span class="string">"--"</span>, label=<span
class="string">f"national mean = {all_wages.mean():.0f} ALL"</span>)
plt.axvline(32784, color=<span class="string">"orange"</span>,
linestyle=<span class="string">"--"</span>, label=<span
class="string">"hospitality avg (2022) = 32,784 ALL"</span>)
plt.title(<span class="string">"RQ2: National Wage Distribution vs
Hospitality Sector (ALL)"</span>) plt.xlabel(<span
class="string">"Monthly Gross Wage (ALL)"</span>) plt.ylabel(<span
class="string">"Frequency"</span>) plt.legend() plt.tight_layout()
plt.show()

In \[7\]:

<span class="comment">\# probability modeling: what's the probability a
national worker \# earns AT OR BELOW the hospitality wage? \# basically
asking: how far into the left tail is hospitality?</span> mu_w =
all_wages.mean() sigma_w = all_wages.std() p_below = norm.cdf(32784,
mu_w, sigma_w) print(<span class="string">f"P(wage \<= 32,784 ALL) =
{p_below:.4f}"</span>) print(<span class="string">f"hospitality workers
are in the bottom {p_below\*100:.1f}% of earners"</span>) x =
np.linspace(40000, 120000, 300) fig, (ax1, ax2) = plt.subplots(1, 2,
figsize=(10, 4)) ax1.plot(x, norm.pdf(x, mu_w, sigma_w), color=<span
class="string">"steelblue"</span>, linewidth=2) ax1.axvline(32784,
color=<span class="string">"orange"</span>, linestyle=<span
class="string">"--"</span>, label=<span class="string">"hospitality
avg"</span>) ax1.axvline(mu_w, color=<span class="string">"red"</span>,
linestyle=<span class="string">"--"</span>, label=<span
class="string">"national avg"</span>) ax1.set_title(<span
class="string">"RQ2: Wage PDF"</span>) ax1.set_xlabel(<span
class="string">"Monthly Wage (ALL)"</span>) ax1.legend() ax2.plot(x,
norm.cdf(x, mu_w, sigma_w), color=<span
class="string">"steelblue"</span>, linewidth=2) ax2.axvline(32784,
color=<span class="string">"orange"</span>, linestyle=<span
class="string">"--"</span>, label=<span class="string">"hospitality
avg"</span>) ax2.set_title(<span class="string">"RQ2: Wage CDF"</span>)
ax2.set_xlabel(<span class="string">"Monthly Wage (ALL)"</span>)
ax2.legend() plt.tight_layout() plt.show() <span class="comment">\#
hypothesis test: one sample t -- is national avg significantly above
32784? \# H0: national avg wage = hospitality wage (32,784) \# H1:
national avg is significantly higher</span> t_stat, p_val =
ttest_1samp(all_wages, 32784) print(<span class="string">f"\nt-stat:
{t_stat:.4f} \| p-value: {p_val:.6f}"</span>) print(<span
class="string">"result: REJECT H0 -- national wages significantly above
hospitality level"</span>) print(<span class="string">"interpretation:
boom not lifting hospitality wages to match the economy"</span>)

P(wage \<= 32,784 ALL) = 0.0000 hospitality workers are in the bottom
0.0% of earners t-stat: 18.8545 \| p-value: 0.000000 result: REJECT H0
-- national wages significantly above hospitality level interpretation:
boom not lifting hospitality wages to match the economy

In \[8\]:

<span class="comment">\##
────────────────────────────────────────────────────────── \## RQ3: Real
Wage Growth vs Inflation \## did nominal wage gains actually outpace
inflation? \## or are workers losing purchasing power in real terms? \##
────────────────────────────────────────────────────────── \# need to
compute year-over-year nominal </span>